# Milestone 6D: Real-Time Structured Streaming & Anomaly Detection

Demonstrates real-time transaction ingestion, watermarking, tumbling window aggregations, and near-real-time AML alerts using **Spark Structured Streaming**.

### Streaming Architecture:
1. **File Stream Source**: Monitors `data/stream/incoming/` micro-batches.
2. **Event-Time Watermark**: 10-minute tolerance for late-arriving events.
3. **5-Minute Tumbling Windows**: Computes rolling transaction count, unique senders, and volume.
4. **Anomaly Rule Filter**: Flags rapid fan-in signatures.
5. **Sinks**: Interactive console output and persistent Parquet sink.


In [1]:
import os
import sys
import time
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.streaming_pipeline import create_spark_session, StreamingAnomalyDetector
from src.stream_simulator import simulate_stream

spark = create_spark_session('Databricks-M6D-Streaming')
detector = StreamingAnomalyDetector(spark)
print('Incoming directory:', detector.incoming_dir)
print('Output directory:  ', detector.output_dir)


C:\Projects\Mule-tiplier Extended\chain-reaction-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Incoming directory: C:\Projects\Mule-tiplier Extended\notebooks\data\stream\incoming
Output directory:   C:\Projects\Mule-tiplier Extended\notebooks\data\stream\output\alerts


## 1. Emit Simulated Micro-Batches
Emit 3 sample micro-batches of transactions into the streaming incoming directory.


In [2]:
source_csv = '../data/processed/processed_df.csv'
if not os.path.exists(source_csv):
    source_csv = '../data/raw/HI-Small_Trans.csv'

simulate_stream(source_csv, detector.incoming_dir, batch_size=200, delay_seconds=0.5, max_batches=3)
print('Sample micro-batches successfully generated.')


2026-10-29 12:10:53,272 [INFO] src.stream_simulator — Reading source dataset from '../data/processed/processed_df.csv'...


2026-10-29 12:10:53,313 [INFO] src.stream_simulator — Source loaded: 1200 total rows. Emitting micro-batches of 200 rows to 'C:\Projects\Mule-tiplier Extended\notebooks\data\stream\incoming'


2026-10-29 12:10:53,323 [INFO] src.stream_simulator — [12:10:53] Emitted Batch 001: 200 transactions -> txn_stream_1793256053_9ac83d.csv


2026-10-29 12:10:53,830 [INFO] src.stream_simulator — [12:10:53] Emitted Batch 002: 200 transactions -> txn_stream_1793256053_675c49.csv


2026-10-29 12:10:54,349 [INFO] src.stream_simulator — [12:10:54] Emitted Batch 003: 200 transactions -> txn_stream_1793256054_b71938.csv


2026-10-29 12:10:54,852 [INFO] src.stream_simulator — Reached configured max_batches limit (3). Simulation terminating.


2026-10-29 12:10:54,854 [INFO] src.stream_simulator — Simulation completed. Emitted 3 total batches.


Sample micro-batches successfully generated.


## 2. Start Structured Streaming Engine
Process micro-batches with tumbling 5-minute event windows and evaluate anomaly thresholds.


In [3]:
queries = detector.start_pipeline(
    enable_console=True,
    enable_parquet=True,
    trigger_interval='2 seconds',
    timeout_seconds=10,
)
print('Streaming execution completed.')
spark.stop()


2026-10-29 12:10:54,862 [INFO] src.streaming_pipeline — Initializing Structured Stream reader from 'C:\Projects\Mule-tiplier Extended\notebooks\data\stream\incoming'...


2026-10-29 12:10:57,289 [INFO] src.streaming_pipeline — Starting Console Sink query...


2026-10-29 12:10:58,299 [INFO] src.streaming_pipeline — Starting Parquet Storage Sink query at 'C:\Projects\Mule-tiplier Extended\notebooks\data\stream\output\alerts'...


2026-10-29 12:10:58,973 [INFO] src.streaming_pipeline — Streaming pipeline launched with 2 active queries.


2026-10-29 12:10:58,974 [INFO] src.streaming_pipeline — Running streaming queries for 10 seconds...


2026-10-29 12:11:08,980 [INFO] src.streaming_pipeline — Timeout elapsed. Stopping queries...


Streaming execution completed.
